# 06 – Zug-Druck-Balkenelemente

Dieses Notebook erweitert die Matrixsteifigkeitsmethode vom reinen **Stabelement** (nur Normalkraft)
auf das **Zug-Druck-Balkenelement** (Normalkraft **und** Biegung).

## Unterschied zum Stabelement

| | Stabelement | Zug-Druck-Balkenelement |
|---|---|---|
| Freiheitsgrade pro Knoten | 2 ($u$, $v$) | 3 ($u$, $v$, $\phi$) |
| Lokale ESM | $2 \times 2$ | $6 \times 6$ |
| Globale ESM | $4 \times 4$ | $6 \times 6$ |
| Schnittgrössen | $N$ | $N$, $V$, $M$ |

## Lokale Elementsteifigkeitsmatrix

$$\underline{\underline{k}}^e_\text{lok} = \frac{E}{L^3}
\begin{pmatrix}
 AL^2  &  0       &  0        & -AL^2  &  0        &  0       \\
 0     & 12I_z    &  6I_z L   &  0     & -12I_z    &  6I_z L  \\
 0     &  6I_z L  &  4I_z L^2 &  0     & -6I_z L   &  2I_z L^2\\
-AL^2  &  0       &  0        &  AL^2  &  0        &  0       \\
 0     &-12I_z    & -6I_z L   &  0     &  12I_z    & -6I_z L  \\
 0     &  6I_z L  &  2I_z L^2 &  0     & -6I_z L   &  4I_z L^2
\end{pmatrix}$$

mit $A$ = Querschnittsfläche [mm²], $I_z$ = Flächenträgheitsmoment [mm⁴], $L$ = Elementlänge [mm].

## Transformationsmatrix

$$\underline{\underline{T}} =
\begin{pmatrix}
 c &  s & 0 & 0 &  0 & 0 \\
-s &  c & 0 & 0 &  0 & 0 \\
 0 &  0 & 1 & 0 &  0 & 0 \\
 0 &  0 & 0 & c &  s & 0 \\
 0 &  0 & 0 &-s &  c & 0 \\
 0 &  0 & 0 & 0 &  0 & 1
\end{pmatrix}, \qquad
\underline{\underline{K}}^e = \underline{\underline{T}}^\top\,\underline{\underline{k}}^e_\text{lok}\,\underline{\underline{T}}$$

## Geänderte Funktionen gegenüber `fem_core.py`

Die vier Kernfunktionen sind **direkt in diesem Notebook** definiert (kein Import aus `fem_core`).
In den Funktionen unten gilt:
- Der ursprüngliche **Stab-Code** ist auskommentiert und gekennzeichnet.
- **Neuer Balken-Code** ist mit #neu# am Zeilenende markiert.

> **Nur in Google Colab noetig:** Die folgende Zelle klont das Repository und macht die Bibliotheksdateien (, ) verfuegbar. Lokal oder in JupyterLite kann sie uebersprungen werden.

In [ ]:
import os, sys
if not os.path.exists("FEM"):
    !git clone --depth=1 -q https://github.com/Boscij/FEM.git
if "FEM/content" not in sys.path:
    sys.path.insert(0, "FEM/content")

In [ ]:
import numpy as np
from fem_post import print_results_beam, plot_results_beam

## Input

Gleiches Fachwerk wie in Notebook 04 – jetzt als Balkentragwerk.
Querschnitte enthalten zusaetzlich $ [mm⁴]: 

Knoten 2 (Index 1) ist eine **Einspannung** (alle drei Freiheitsgrade gesperrt:  = u_y = arphi = 0$).



In [ ]:
# Knotenkoordinaten [X, Y] in mm
nodal_coordinates = np.array([
    [   0.0,    0.0],
    [1000.0,    0.0],
    [1000.0, 1000.0],
    [2000.0, 1000.0],
])

# Elemente: [Knoten_i, Knoten_j, section_key]
elements = [
    [0, 1, "section 1"],
    [0, 2, "section 2"],
    [1, 2, "section 3"],
    [1, 3, "section 4"],
    [2, 3, "section 5"],
]

# Materialien: E-Modul [MPa = N/mm2]
materials = {"steel": [210000.0]}

# Querschnitte: [A [mm2], #neu# Iz [mm4], material_key]
sections = {
    "section 1": [ 15.00,  1000.0, "steel"],
    "section 2": [ 28.28,  2828.0, "steel"],
    "section 3": [ 10.00,  1000.0, "steel"],
    "section 4": [ 56.56,  5656.0, "steel"],
    "section 5": [ 10.00,  1000.0, "steel"],
}

# Randbedingungen: [Knoten (0-basiert), Achse (0=x, 1=y, 2=phi), vorgegebene Verschiebung]
constraints = [
    [0, 0, 0.0],   # Knoten 1: u_x = 0   (Festlager, x)
    [0, 1, 0.0],   # Knoten 1: u_y = 0   (Festlager, y)
    [1, 0, 0.0],   # Knoten 2: u_x = 0   (Einspannung)
    [1, 1, 0.0],   # Knoten 2: u_y = 0   (Einspannung)
    [1, 2, 0.0],   # NEU! Knoten 2: phi = 0   (Einspannung)
]

# Lasten: [Knoten (0-basiert), Achse (0=x, 1=y, 2=phi), Kraft [N] oder Moment [Nmm]]
loads = [
    [3, 1, -1000.0],   # Knoten 4: F_y = -1000 N
]

## Geänderte Funktionen

### 1. Elementsteifigkeitsmatrix `element_stiffness_matrix`

**Was ändert sich:**
- Neues Argument `Iz` (Flächenträgheitsmoment)
- Lokale ESM: $6 \times 6$ statt $2 \times 2$, enthält Biegeterme $EI_z$
- Transformationsmatrix $\underline{\underline{T}}$: $6 \times 6$ statt $4 \times 4$, dritte Zeile/Spalte für Rotation $\phi$

**Alt (Stabelement):** `k_lokal = (EA/L) * [[1,-1],[-1,1]]`, $\underline{\underline{T}} \in \mathbb{R}^{2\times4}$

**Neu (Balkenelement):** $\underline{\underline{k}}_\text{lok} \in \mathbb{R}^{6\times6}$, $\underline{\underline{T}} \in \mathbb{R}^{6\times6}$

In [ ]:
def element_stiffness_matrix(E, A, Iz, xy_e):
    """Globale ESM K^e fuer ein 2D Zug-Druck-Balkenelement (6x6)."""
    dx = xy_e[1, 0] - xy_e[0, 0]
    dy = xy_e[1, 1] - xy_e[0, 1]
    L  = np.sqrt(dx**2 + dy**2)
    c  = dx / L
    s  = dy / L

    # Stab:   k_lokal = (E*A / L) * np.array([[1, -1], [-1, 1]])
    k_lokal = (E / L**3) * np.array([                                                   #neu#
        [ A*L**2,      0,           0,         -A*L**2,      0,           0         ],  #neu#
        [ 0,          12*Iz,        6*Iz*L,     0,          -12*Iz,       6*Iz*L    ],  #neu#
        [ 0,           6*Iz*L,      4*Iz*L**2,  0,           -6*Iz*L,     2*Iz*L**2 ],  #neu#
        [-A*L**2,      0,           0,          A*L**2,       0,           0         ], #neu#
        [ 0,         -12*Iz,       -6*Iz*L,     0,           12*Iz,       -6*Iz*L   ],  #neu#
        [ 0,           6*Iz*L,      2*Iz*L**2,  0,           -6*Iz*L,     4*Iz*L**2 ],  #neu#
    ])                                                                                  

    # Stab:   T = np.array([[c, s, 0, 0],
    #                        [0, 0, c, s]])
    T = np.array([                      #neu#
        [ c,  s,  0,  0,  0,  0],       #neu#
        [-s,  c,  0,  0,  0,  0],       #neu#
        [ 0,  0,  1,  0,  0,  0],       #neu#
        [ 0,  0,  0,  c,  s,  0],       #neu#
        [ 0,  0,  0, -s,  c,  0],       #neu#
        [ 0,  0,  0,  0,  0,  1],       #neu#
    ])

    return T.T @ k_lokal @ T

### 2. Koinzidenztabelle `incidence_table`

**Was ändert sich:**
- Pro Knoten jetzt **3 Freiheitsgrade** statt 2: $(u, v, \phi)$
- DOF-Indizes: $[3i,\; 3i{+}1,\; 3i{+}2,\; 3j,\; 3j{+}1,\; 3j{+}2]$ statt $[2i,\; 2i{+}1,\; 2j,\; 2j{+}1]$

In [ ]:
def incidence_table(elements):
    """DOF-Indizes je Element: [3i, 3i+1, 3i+2, 3j, 3j+1, 3j+2]."""
    conn = np.array([[e[0], e[1]] for e in elements], dtype=int)
    return np.vstack((
        # Stab: 2 * conn[:, 0],
        3 * conn[:, 0],          # u_i    #neu#
        # Stab: 2 * conn[:, 0] + 1,
        3 * conn[:, 0] + 1,      # v_i    #neu#
        3 * conn[:, 0] + 2,      # phi_i  #neu#
        # Stab: 2 * conn[:, 1],
        3 * conn[:, 1],          # u_j    #neu#
        # Stab: 2 * conn[:, 1] + 1,
        3 * conn[:, 1] + 1,      # v_j    #neu#
        3 * conn[:, 1] + 2,      # phi_j  #neu#
    )).T

dof_table = incidence_table(elements)
print(dof_table)

### 3. Assemblierung `assemble_K`

**Was ändert sich:**
- Aufruf von `element_stiffness_matrix` mit neuer Signatur `(E, A, Iz, xy_e)`
- `sections` liefert jetzt drei Werte: `A, Iz, mat_key`

In [ ]:
def assemble_K(nodal_coordinates, elements, sections, materials):
    """Assembliert die globale Steifigkeitsmatrix K."""
    dofs = incidence_table(elements)
    ndof = int(np.max(dofs) + 1)
    K    = np.zeros((ndof, ndof))

    for e, (i, j, sec_key) in enumerate(elements):
        # Stab:   A, mat_key = sections[sec_key]
        A, Iz, mat_key = sections[sec_key]                    #neu#
        E    = materials[mat_key][0]
        xy_e = nodal_coordinates[[i, j], :]
        # Stab:   Ke = element_stiffness_matrix(E * A, xy_e)
        Ke   = element_stiffness_matrix(E, A, Iz, xy_e)      #neu#
        idx  = dofs[e]
        K[np.ix_(idx, idx)] += Ke

    return K

### 4. Gleichungssystem lösen `solve_system`

**Was ändert sich:**
- DOF-Index: `3 * node + axis` statt `2 * node + axis`
- Achse 2 entspricht jetzt dem Rotations-DOF $\phi$

In [ ]:
def solve_system(K, constraints, loads):
    """Loest K * U = F mit vorgeschriebenen Verschiebungen und Knotenlasten."""
    ndof = K.shape[0]

    fixed        = np.zeros(ndof, dtype=bool)
    U_prescribed = []
    for node, axis, val in constraints:
        # Stab:   dof = 2 * int(node) + int(axis)
        dof = 3 * int(node) + int(axis)   #neu#
        fixed[dof] = True
        U_prescribed.append(val)
    U_prescribed = np.array(U_prescribed, dtype=float)
    free = ~fixed

    F = np.zeros(ndof)
    for node, axis, val in loads:
        # Stab:   dof = 2 * int(node) + int(axis)
        dof = 3 * int(node) + int(axis)   #neu#
        F[dof] = val

    K_FF = K[np.ix_(free,  free)]
    K_FU = K[np.ix_(free,  fixed)]
    F_F  = F[free]
    U_free = np.linalg.solve(K_FF, F_F - K_FU @ U_prescribed)

    U = np.zeros(ndof)
    U[fixed] = U_prescribed
    U[free]  = U_free

    K_UF = K[np.ix_(fixed, free)]
    K_UU = K[np.ix_(fixed, fixed)]
    F[fixed] = K_UF @ U_free + K_UU @ U_prescribed

    return U, F, fixed

## Berechnung

In [ ]:
K        = assemble_K(nodal_coordinates, elements, sections, materials)
U, F, _  = solve_system(K, constraints, loads)

## Postprocessing

Aus den lokalen Verschiebungen $\boldsymbol{u}^e_\text{lok} = \underline{\underline{T}}\,\boldsymbol{U}^e$
folgen die Schnittgrössen am Element-Anfang (Knoten $i$) und -Ende (Knoten $j$):

$$\boldsymbol{f}^e_\text{lok} = \underline{\underline{k}}^e_\text{lok}\,\boldsymbol{u}^e_\text{lok}
\quad\Rightarrow\quad
N,\; V,\; M \text{ an beiden Enden}$$

In [ ]:
def postprocessing(U, nodal_coordinates, elements, sections, materials):
    """Berechnet Normalkraft N, Querkraft V und Biegemoment M je Element-Ende."""
    n_el = len(elements)
    N = np.zeros((n_el, 2))   # Normalkraft  [N]   an Knoten i und j
    V = np.zeros((n_el, 2))   # Querkraft    [N]   an Knoten i und j
    M = np.zeros((n_el, 2))   # Biegemoment  [Nmm] an Knoten i und j

    for e, (i, j, sec_key) in enumerate(elements):
        A_e, Iz_e, mat_key = sections[sec_key]
        E_e = materials[mat_key][0]

        # Elementgeometrie
        xy_e = nodal_coordinates[[i, j], :]
        dx = xy_e[1, 0] - xy_e[0, 0]
        dy = xy_e[1, 1] - xy_e[0, 1]
        L  = np.sqrt(dx**2 + dy**2)
        c  = dx / L;  s = dy / L

        # Transformationsmatrix (global -> lokal)
        T = np.array([
            [ c,  s,  0,  0,  0,  0],
            [-s,  c,  0,  0,  0,  0],
            [ 0,  0,  1,  0,  0,  0],
            [ 0,  0,  0,  c,  s,  0],
            [ 0,  0,  0, -s,  c,  0],
            [ 0,  0,  0,  0,  0,  1],
        ])

        # Lokale ESM
        k_lok = (E_e / L**3) * np.array([
            [ A_e*L**2,    0,             0,           -A_e*L**2,    0,             0           ],
            [ 0,          12*Iz_e,        6*Iz_e*L,    0,           -12*Iz_e,       6*Iz_e*L    ],
            [ 0,           6*Iz_e*L,      4*Iz_e*L**2, 0,            -6*Iz_e*L,     2*Iz_e*L**2 ],
            [-A_e*L**2,    0,             0,            A_e*L**2,    0,             0           ],
            [ 0,         -12*Iz_e,       -6*Iz_e*L,    0,            12*Iz_e,      -6*Iz_e*L   ],
            [ 0,           6*Iz_e*L,      2*Iz_e*L**2, 0,            -6*Iz_e*L,     4*Iz_e*L**2 ],
        ])

        # Globale Element-Verschiebungen extrahieren und in lokale umrechnen
        U_e = U[[3*i, 3*i+1, 3*i+2, 3*j, 3*j+1, 3*j+2]]
        u_l = T @ U_e          # lokale Verschiebungen: [u_i, v_i, phi_i, u_j, v_j, phi_j]
        f_l = k_lok @ u_l      # lokale Schnittkraefte an beiden Elementenden

        # Vorzeichenkonvention: Kraefte/Momente am Knoten i zeigen in negative lok. Richtung
        N[e] = [-f_l[0],  f_l[3]]   # Normalkraft
        V[e] = [ f_l[1], -f_l[4]]   # Querkraft
        M[e] = [ f_l[2], -f_l[5]]   # Biegemoment

    return N, V, M

N, V, M = postprocessing(U, nodal_coordinates, elements, sections, materials)

## Ergebnisse

> Der Code fuer Ausgabe und Visualisierung ist in fem_post.py ausgelagert und nicht Teil dieser Übung.

In [ ]:
print_results_beam(U, F, N, V, M, nodal_coordinates, constraints, elements)

## Visualisierung

In [ ]:
plot_results_beam(nodal_coordinates, elements, constraints, loads, U, N, scale=20)